<a href="https://colab.research.google.com/github/Aditya-Raj-Kaushik/Vision-Transformer/blob/main/Vision_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import random

In [2]:
torch.__version__

'2.10.0+cu128'

In [3]:
torchvision.__version__

'0.25.0+cu128'

In [4]:
!nvidia-smi

Mon May 25 15:52:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
torch.cuda.is_available()

True

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
print(f"Using Device: {device}")

Using Device: cuda


In [8]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

In [9]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [11]:
train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

100%|██████████| 170M/170M [00:04<00:00, 36.2MB/s]


In [12]:
test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform
)

In [13]:
train_data

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [14]:
len(train_data)

50000

In [15]:
len(test_data)

10000

In [16]:
train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=True)

In [17]:
print(f"Dataloader: {train_loader, test_loader}")
print(f"Length of Trainloader: {len(train_loader)}")
print(f"Length of Testloader: {len(test_loader)}")

Dataloader: (<torch.utils.data.dataloader.DataLoader object at 0x7b6bccdea150>, <torch.utils.data.dataloader.DataLoader object at 0x7b6bcd223230>)
Length of Trainloader: 391
Length of Testloader: 79


In [24]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()

        self.patch_size = patch_size

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)

        x = x.flatten(2)

        x = x.transpose(1, 2)

        return x


In [25]:
class MLP(nn.Module):
  def __init__(self, in_features, hidden_features, out_features, drop_rate=0.1):
    super().__init__()
    self.fc1 = nn.Linear(in_features=in_features, out_features=hidden_features)
    self.fc2 = nn.Linear(in_features=hidden_features, out_features=in_features)
    self.drop = nn.Dropout(drop_rate)

    def forward(self, x):
      x = self.dropout(F.gelu(self.fc1(x)))
      x = self.fc2(x)
      return x

In [26]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=drop_rate,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x)
        )[0]

        x = x + self.mlp(self.norm2(x))

        return x

In [27]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size,
        patch_size,
        in_channels,
        num_classes,
        embed_dim,
        depth,
        num_heads,
        mlp_dim,
        drop_rate
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size,
            patch_size,
            in_channels,
            embed_dim
        )

        self.encoder = nn.Sequential(
            *[
                TransformerEncoderLayer(
                    embed_dim,
                    num_heads,
                    mlp_dim,
                    drop_rate
                )
                for _ in range(depth)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)

        x = self.encoder(x)

        x = self.norm(x)

        cls_token = x[:, 0]

        return self.head(cls_token)

In [28]:
model = VisionTransformer(
    IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES, EMBED_DIM, DEPTH, NUM_HEADS, MLP_DIM, DROP_RATE
).to(device)

In [29]:
model

VisionTransformer(
  (patch_embed): PatchEmbedding(
    (proj): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
  )
  (encoder): Sequential(
    (0): TransformerEncoderLayer(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=256, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=256, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
    (1): TransformerEncoderLayer(
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
 

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [31]:
criterion

CrossEntropyLoss()

In [32]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0
)

In [34]:
def train(model, loader, optimizer, criterion):
    model.train()

    total_loss, correct = 0, 0

    for x,y in loader:
      x,y = x.to(device), y.to(device)

      optimizer.zero_grad()

      y_pred = model(x)

      loss = criterion(y_pred, y)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()*x.size(0)

      correct += (y_pred.argmax(1) == y).sum().item()

    return total_loss/len(loader.dataset), correct/len(loader.dataset)

In [35]:
def evaluate(model, loader):
  model.eval()
  correct = 0
  with torch.inference_mode():
    for x,y in loader:
      x,y = x.to(device), y.to(device)
      y_pred = model(x)
      correct += (y_pred.argmax(dim=1) == y).sum().item()
  return correct/len(loader.dataset)